### M1 모델 생성 및 성능평가

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from scipy.stats import pearsonr, spearmanr


# =========================================================
# 1. 데이터 불러오기
# =========================================================

file_path = (
    r"C:\Users\user\Desktop\AI경진대회\[AI경진대회]DATA\전처리\[새로해보는2]인천_분기x읍면동x업태_다음분기폐업률_전처리_가통합.csv"
)

df = pd.read_csv(file_path)

print("전체 데이터:", df.shape)
print(df["기준분기"].value_counts().sort_index())


# =========================================================
# 2. M1 Feature
#    M0 + 시간순번
# =========================================================

features = [

    # -------------------------
    # 시간
    # -------------------------
    "분기번호",              

    # -------------------------
    # 업태
    # -------------------------
    "업태구분_2진수",

    # -------------------------
    # 읍면동 × 업태 사업체 구성
    # -------------------------
    "읍면동업태_영업점포수",
    "평균업력개월",
    "업력1년미만_점포비중",
    "평균시설규모",
    "중앙시설규모",
    "소규모점포비중",
    "다중이용업소비중",

    # -------------------------
    # 법정동 수요 / 경쟁 환경
    # -------------------------
    "법정동_실제총인구",
    "법정동_인구증감률_QoQ",
    "법정동_인구증감률_YoY",
    "법정동_인구천명당_전체점포수",
    "법정동_인구천명당_동일업태점포수",

    # -------------------------
    # 읍면동 × 업태 상권 상태
    # -------------------------
    "읍면동업태_분기개업수",
    "읍면동업태_분기폐업수",
    "읍면동업태_점포비중",
    "읍면동업태_직전분기말점포수",
    "읍면동업태_개업률",
    "읍면동업태_폐업률",
    "읍면동업태_점포수증감률",
    "읍면동업태_폐업률_lag1",
    "읍면동업태_폐업률_lag4",
    "읍면동업태_폐업률_4Q평균",
    "읍면동업태_폐업률_전분기대비증감",
]

target = "다음분기_폐업률"

print("\nM1 Feature 수:", len(features))


# =========================================================
# 3. Train / Test
# Train : 2023Q1 ~ 2025Q4
# Test  : 2026Q1 ~ 2026Q2
# =========================================================

train_df = df[df["기준연도"] <= 2025].copy()
test_df = df[df["기준연도"] == 2026].copy()

X_train = train_df[features].copy()
y_train = train_df[target].astype(float)

X_test = test_df[features].copy()
y_test = test_df[target].astype(float)

print("Train:", len(train_df))
print("Test :", len(test_df))


# =========================================================
# 4. Sample Weight
# sqrt(읍면동업태 영업점포수)
# =========================================================

train_weight = np.sqrt(
    train_df["읍면동업태_영업점포수"].astype(float)
)

test_weight = np.sqrt(
    test_df["읍면동업태_영업점포수"].astype(float)
)


# =========================================================
# 5. LightGBM
# M0와 동일한 하이퍼파라미터
# =========================================================

model = lgb.LGBMRegressor(
    objective="regression_l1",

    learning_rate=0.05,
    num_leaves=15,
    min_child_samples=70,

    reg_alpha=0.1,
    reg_lambda=1.0,

    colsample_bytree=0.9,
    subsample=0.9,

    max_depth=-1,
    n_estimators=120,

    random_state=42,
    verbosity=-1,
    force_col_wise=True,
    n_jobs=4
)


# =========================================================
# 6. 학습
# =========================================================

model.fit(
    X_train,
    y_train,
    sample_weight=train_weight
)


# =========================================================
# 7. 예측
# =========================================================

y_pred = model.predict(X_test)

# 폐업률이므로 0~1 범위
y_pred = np.clip(y_pred, 0, 1)


# =========================================================
# 8. 평가
# =========================================================

mae = mean_absolute_error(
    y_test,
    y_pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)

wmae = np.average(
    np.abs(y_test - y_pred),
    weights=test_weight
)

wrmse = np.sqrt(
    np.average(
        (y_test.to_numpy() - y_pred) ** 2,
        weights=test_weight.to_numpy()
    )
)

pearson = pearsonr(
    y_test,
    y_pred
)[0]

spearman = spearmanr(
    y_test,
    y_pred
)[0]

r2 = r2_score(
    y_test,
    y_pred
)


# =========================================================
# 9. 결과 출력
# =========================================================

print("\n==========================================")
print(" M1 : M0 + 시간순번")
print("==========================================")

print(f"MAE      : {mae:.6f} ({mae*100:.3f}%p)")
print(f"RMSE     : {rmse:.6f} ({rmse*100:.3f}%p)")
print(f"WMAE     : {wmae:.6f} ({wmae*100:.3f}%p)")
print(f"WRMSE    : {wrmse:.6f} ({wrmse*100:.3f}%p)")
print(f"Pearson  : {pearson:.3f}")
print(f"Spearman : {spearman:.3f}")
print(f"R2       : {r2:.3f}")


# =========================================================
# 10. Feature Importance
# =========================================================

importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
})

importance_df = importance_df.sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)

importance_df["Importance_pct"] = (
    importance_df["Importance"]
    / importance_df["Importance"].sum()
    * 100
)

print("\n===== Feature Importance Top 15 =====")
print(
    importance_df.head(15).to_string(index=False)
)


# 시간순번 중요도 별도 확인
print("\n===== 시간순번 중요도 =====")
print(
    importance_df[
        importance_df["Feature"] == "시간순번"
    ].to_string(index=False)
)


# =========================================================
# 11. 테스트 예측 결과 저장
# =========================================================

result = test_df[
    [
        "기준분기",
        "예측대상분기",
        "군구명_개편후",
        "읍면동_개편후",
        "업태구분명",
        "읍면동업태_영업점포수",
        "다음분기_폐업수",
        "다음분기_폐업률"
    ]
].copy()

result["예측_다음분기폐업률"] = y_pred

result["절대오차"] = np.abs(
    result["다음분기_폐업률"]
    - result["예측_다음분기폐업률"]
)

result.to_csv(
    "M1_시간순번추가_2026Q1_Q2_예측결과.csv",
    index=False,
    encoding="utf-8-sig"
)


# =========================================================
# 12. 성능 저장
# =========================================================

metrics_df = pd.DataFrame({
    "Model": ["M1_M0+시간순번"],
    "MAE": [mae],
    "RMSE": [rmse],
    "WMAE": [wmae],
    "WRMSE": [wrmse],
    "Pearson": [pearson],
    "Spearman": [spearman],
    "R2": [r2]
})

metrics_df.to_csv(
    "M1_시간순번추가_성능평가.csv",
    index=False,
    encoding="utf-8-sig"
)

importance_df.to_csv(
    "M1_시간순번추가_FeatureImportance.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n저장 완료")

### M2 모델 생성

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from scipy.stats import pearsonr, spearmanr


# =========================================================
# 1. 데이터 불러오기
# =========================================================

file_path = (
    r"C:\Users\user\Desktop\AI경진대회\[AI경진대회]DATA\전처리\[새로해보는3]인천_분기x읍면동x업태_총매출추가_2023추정.csv"
)

df = pd.read_csv(file_path)

print("전체 데이터:", df.shape)
print(df["기준분기"].value_counts().sort_index())


# =========================================================
# 2. 피처 및 타겟 설정
# =========================================================

features = [
    # 시간 (시간순번 제거)
    "분기번호",
    
    # 업태
    "업태구분_2진수",
    
    # 읍면동 × 업태 사업체 구성
    "읍면동업태_영업점포수",
    "평균업력개월",
    "업력1년미만_점포비중",
    "평균시설규모",
    "중앙시설규모",
    "소규모점포비중",
    "다중이용업소비중",
    
    # 법정동 수요 / 경쟁 환경
    "법정동_실제총인구",
    "법정동_인구증감률_QoQ",
    "법정동_인구증감률_YoY",
    "법정동_인구천명당_전체점포수",
    "법정동_인구천명당_동일업태점포수",
    
    # 읍면동 × 업태 상권 상태
    "읍면동업태_분기개업수",
    "읍면동업태_분기폐업수",
    "읍면동업태_점포비중",
    "읍면동업태_직전분기말점포수",
    "읍면동업태_개업률",
    "읍면동업태_폐업률",
    "읍면동업태_점포수증감률",
    "읍면동업태_폐업률_lag1",
    "읍면동업태_폐업률_lag4",
    "읍면동업태_폐업률_4Q평균",
    "읍면동업태_폐업률_전분기대비증감",

    "NICE_총매출_만원"
]

# 타겟을 폐업수(건수)로 변경
target_count = "다음분기_폐업수"
# 최종 평가는 여전히 폐업률로 진행하기 위해 별도 저장
target_rate_eval = "다음분기_폐업률" 

print("\nFeature 수:", len(features))


# =========================================================
# 3. Train / Test 분할
# =========================================================

train_df = df[df["기준연도"] <= 2025].copy()
test_df = df[df["기준연도"] == 2026].copy()

X_train = train_df[features].copy()
y_train_count = train_df[target_count].astype(float)

X_test = test_df[features].copy()
y_test_count = test_df[target_count].astype(float)

# 평가를 위한 실제 폐업률
y_test_rate = test_df[target_rate_eval].astype(float) 

print("Train:", len(train_df))
print("Test :", len(test_df))


# =========================================================
# 4. LightGBM (Poisson 모델)
# =========================================================

model = lgb.LGBMRegressor(
    objective="poisson",       # 건수 예측에 최적화된 포아송 회귀 적용
    learning_rate=0.05,
    num_leaves=15,
    min_child_samples=30,      # 과소적합 방지를 위해 30으로 완화
    reg_alpha=0.1,
    reg_lambda=1.0,
    colsample_bytree=0.9,
    subsample=0.9,
    max_depth=-1,
    n_estimators=120,
    random_state=42,
    verbosity=-1,
    force_col_wise=True,
    n_jobs=4
)


# =========================================================
# 5. 학습
# 포아송 회귀는 값의 크기 자체가 비중 역할을 하므로 Weight 생략 가능
# =========================================================

model.fit(X_train, y_train_count)


# =========================================================
# 6. 예측 및 비율 변환
# =========================================================

# 1) 먼저 다음분기 '폐업수'를 예측
pred_count = model.predict(X_test)

# 2) 예측된 폐업수를 기준 영업점포수로 나누어 '폐업률'로 변환
# 분모가 0인 경우를 방지하기 위해 매우 작은 값(1e-9)을 더해주는 것이 안전할 수 있습니다.
pred_rate = pred_count / (X_test["읍면동업태_영업점포수"] + 1e-9)

# 3) 폐업률은 0~1 사이이므로 클리핑 적용
y_pred_rate = np.clip(pred_rate, 0, 1)


# =========================================================
# 7. 평가 (실제 폐업률 vs 예측 폐업률)
# =========================================================

test_weight = np.sqrt(test_df["읍면동업태_영업점포수"].astype(float))

mae = mean_absolute_error(y_test_rate, y_pred_rate)
rmse = np.sqrt(mean_squared_error(y_test_rate, y_pred_rate))

wmae = np.average(np.abs(y_test_rate - y_pred_rate), weights=test_weight)
wrmse = np.sqrt(np.average((y_test_rate - y_pred_rate) ** 2, weights=test_weight))

pearson = pearsonr(y_test_rate, y_pred_rate)[0]
spearman = spearmanr(y_test_rate, y_pred_rate)[0]
r2 = r2_score(y_test_rate, y_pred_rate)

print("\n==========================================")
print(" 타겟변경: 다음분기_폐업수 예측 후 비율 변환 (시간순번 제외)")
print("==========================================")
print(f"MAE      : {mae:.6f} ({mae*100:.3f}%p)")
print(f"RMSE     : {rmse:.6f} ({rmse*100:.3f}%p)")
print(f"WMAE     : {wmae:.6f} ({wmae*100:.3f}%p)")
print(f"WRMSE    : {wrmse:.6f} ({wrmse*100:.3f}%p)")
print(f"Pearson  : {pearson:.3f}")
print(f"Spearman : {spearman:.3f}")
print(f"R2       : {r2:.3f}")


# =========================================================
# 8. Feature Importance
# =========================================================

importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
})

importance_df = importance_df.sort_values("Importance", ascending=False).reset_index(drop=True)
importance_df["Importance_pct"] = (importance_df["Importance"] / importance_df["Importance"].sum() * 100)

print("\n===== Feature Importance Top 15 =====")
print(importance_df.head(15).to_string(index=False))


# =========================================================
# 9. 테스트 예측 결과 저장
# =========================================================

result = test_df[
    [
        "기준분기",
        "예측대상분기",
        "군구명_개편후",
        "읍면동_개편후",
        "업태구분명",
        "읍면동업태_영업점포수",
        "다음분기_폐업수",
        "다음분기_폐업률"
    ]
].copy()

# ★ 추가: 모델이 직접 예측한 '폐업수' 매핑
result["예측_다음분기폐업수"] = pred_count

# 최종 예측률 매핑
result["예측_다음분기폐업률"] = y_pred_rate
result["절대오차"] = np.abs(result["다음분기_폐업률"] - result["예측_다음분기폐업률"])

result.to_csv(
    "Count예측_시간순번제외_비율변환_2026Q1_Q2_예측결과3.csv",
    index=False,
    encoding="utf-8-sig"
)
print("\n저장 완료")

### M2 모델 성능평가

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from scipy.stats import pearsonr, spearmanr

# =========================================================
# 건수(Count) 기준 가중치 및 성능 평가 지표 계산
# (test_df, y_test_count, pred_count 변수가 메모리에 있다고 가정)
# =========================================================

actual_count = y_test_count
predicted_count = pred_count

# 가중치 다시 정의
test_weight = np.sqrt(test_df["읍면동업태_영업점포수"].astype(float))

# 기본 지표
mae_count = mean_absolute_error(actual_count, predicted_count)
rmse_count = np.sqrt(mean_squared_error(actual_count, predicted_count))

# 가중치 적용 지표 (WMAE, WRMSE)
wmae_count = np.average(np.abs(actual_count - predicted_count), weights=test_weight)
wrmse_count = np.sqrt(np.average((actual_count - predicted_count) ** 2, weights=test_weight))

# 상관계수 및 결정계수
pearson_count = pearsonr(actual_count, predicted_count)[0]
spearman_count = spearmanr(actual_count, predicted_count)[0]
r2_count = r2_score(actual_count, predicted_count)

# =========================================================
# 평가 결과 화면 출력
# =========================================================

print("\n==========================================")
print(" 성능 평가 결과 (건수 예측 모델 - 폐업수 및 가중치 적용)")
print("==========================================")
print(f"MAE      : {mae_count:.6f} 건")
print(f"RMSE     : {rmse_count:.6f} 건")
print(f"WMAE     : {wmae_count:.6f} 건")
print(f"WRMSE    : {wrmse_count:.6f} 건")
print(f"Pearson  : {pearson_count:.3f}")
print(f"Spearman : {spearman_count:.3f}")
print(f"R2       : {r2_count:.3f}")

# =========================================================
# 평가 결과 CSV 파일 저장
# =========================================================

metrics_count_df = pd.DataFrame({
    "Model": ["Count예측_시간순번제외_WMAE포함"],
    "MAE": [mae_count],
    "RMSE": [rmse_count],
    "WMAE": [wmae_count],
    "WRMSE": [wrmse_count],
    "Pearson": [pearson_count],
    "Spearman": [spearman_count],
    "R2": [r2_count]
})

metrics_count_df.to_csv(
    "Count예측_시간순번제외_건수평가_결과3.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n성능 평가 지표 저장 완료: Count예측_시간순번제외_건수평가_결과3.csv")

### M2 SHAP 값추출

In [ ]:
# =========================================================
# SHAP 분석
# =========================================================

# 설치가 안 되어 있다면 최초 1회
# !pip install shap

import shap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# =========================================================
# 1. SHAP Explainer 생성
# =========================================================

explainer = shap.TreeExplainer(model)


# =========================================================
# 2. SHAP 값 계산
# =========================================================
# 현재 학습 데이터 전체를 기준으로 전역 중요도 계산

shap_values = explainer.shap_values(X_train)

print("SHAP 값 shape:", np.array(shap_values).shape)
print("X_train shape :", X_train.shape)


# =========================================================
# 3. 변수별 mean(|SHAP|) 계산
# =========================================================
# 각 변수의 SHAP 절댓값을 모든 관측치에 대해 평균
# → 값이 클수록 모델 예측에 평균적으로 큰 영향을 준 변수

mean_abs_shap = np.abs(shap_values).mean(axis=0)

shap_importance_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Mean_ABS_SHAP": mean_abs_shap
})

shap_importance_df = (
    shap_importance_df
    .sort_values("Mean_ABS_SHAP", ascending=False)
    .reset_index(drop=True)
)


# =========================================================
# 4. 상대적 중요도(%) 계산
# =========================================================

shap_importance_df["Importance_pct"] = (
    shap_importance_df["Mean_ABS_SHAP"]
    / shap_importance_df["Mean_ABS_SHAP"].sum()
    * 100
)

# 누적 중요도
shap_importance_df["Cumulative_pct"] = (
    shap_importance_df["Importance_pct"].cumsum()
)


print("\n==========================================")
print(" SHAP Feature Importance")
print("==========================================")

print(
    shap_importance_df.to_string(
        index=False,
        formatters={
            "Mean_ABS_SHAP": "{:.6f}".format,
            "Importance_pct": "{:.2f}".format,
            "Cumulative_pct": "{:.2f}".format
        }
    )
)


# =========================================================
# 5. SHAP 중요도 Bar Plot
# =========================================================

shap.summary_plot(
    shap_values,
    X_train,
    plot_type="bar",
    max_display=len(features),
    show=True
)


# =========================================================
# 6. SHAP Summary Plot
# =========================================================
# 변수의 중요도뿐만 아니라
# 변수 값이 높고/낮을 때 예측값에 어떤 영향을 주는지도 확인

shap.summary_plot(
    shap_values,
    X_train,
    max_display=len(features),
    show=True
)


# =========================================================
# 7. SHAP 중요도 CSV 저장
# =========================================================

shap_importance_df.to_csv(
    "Poisson_SHAP_FeatureImportance.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nSHAP 중요도 저장 완료")

### M3 모델 생성 및 SHAP 값 추출

In [ ]:
# ============================================================
# 최종 모델: B2 (Offset Poisson LightGBM) - M3
#   Target : 다음분기_폐업수
#   Offset : log(영업점포수) + c,  c = log(학습구간 평균 폐업률)
#   Train  : 2023Q1 ~ 2025Q4  /  Test : 2026Q1 ~ 2026Q2 (1회 평가)
#
#   주의: 이 스크립트의 결과를 보고 변수/파라미터를 바꿔 다시 돌리지 말 것
#         (테스트가 모델 선택에 섞여 성능 보고의 의미가 사라짐)
# ============================================================
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
from pathlib import Path

from matplotlib import font_manager

# 설치된 한글 폰트 중 첫 번째만 사용 (없는 폰트 경고 방지)
_installed = {f.name for f in font_manager.fontManager.ttflist}
for _font in ["Malgun Gothic", "AppleGothic", "NanumGothic"]:
    if _font in _installed:
        plt.rcParams["font.family"] = _font
        break
plt.rcParams["axes.unicode_minus"] = False

FILE_PATH = os.environ.get(
    "DATA_PATH",
    r"C:\Users\user\Desktop\AI경진대회\[AI경진대회]DATA\전처리\[채택][새로해보는3]인천_분기x읍면동x업태_총매출추가_2023추정.csv",
)
OUT = Path("B2_Final_Results"); OUT.mkdir(exist_ok=True)

N, Y = "읍면동업태_영업점포수", "다음분기_폐업수"

# ---------------- B2 변수 (22개) ----------------
FEATURES = [
    "분기번호", "업태구분_2진수",
    "평균업력개월", "업력1년미만_점포비중",
    "평균시설규모", "중앙시설규모", "소규모점포비중", "다중이용업소비중",
    "법정동_실제총인구", "법정동_인구증감률_QoQ", "법정동_인구증감률_YoY",
    "법정동_인구천명당_전체점포수", "법정동_인구천명당_동일업태점포수",
    "읍면동업태_점포비중",
    "읍면동업태_개업률", "읍면동업태_폐업률", "읍면동업태_점포수증감률",
    "읍면동업태_폐업률_lag1", "읍면동업태_폐업률_lag4",
    "읍면동업태_폐업률_4Q평균", "읍면동업태_폐업률_전분기대비증감",
    "점포당_매출_log",
]

PARAMS = dict(objective="poisson", learning_rate=0.05, num_leaves=15,
              min_child_samples=30, reg_alpha=0.1, reg_lambda=1.0,
              colsample_bytree=0.9, subsample=0.9, subsample_freq=1,
              n_estimators=200, random_state=42, verbosity=-1,
              force_col_wise=True, n_jobs=4)

# ---------------- 데이터 ----------------
df = pd.read_csv(FILE_PATH)
df[N] = pd.to_numeric(df[N], errors="coerce")
assert (df[N] > 0).all(), "영업점포수가 0 이하/결측인 행이 있습니다."

df["점포당_매출_log"] = np.log1p(df["NICE_총매출_만원"] / df[N])
for c in FEATURES:
    df[c] = pd.to_numeric(df[c], errors="coerce").replace([np.inf, -np.inf], np.nan)  # NaN은 그대로 둠

train = df[df["기준연도"] <= 2025].copy()
test = df[df["기준연도"] == 2026].copy()
print(f"Train {train.shape} | Test {test.shape} | 변수 {len(FEATURES)}개")
print("Test 분기:", sorted(test["기준분기"].unique()))

# ---------------- 학습 (Offset Poisson) ----------------
c = np.log(train[Y].sum() / train[N].sum())            # 평균 폐업률의 log
print(f"c = {c:.4f}  (학습구간 평균 폐업률 {np.exp(c):.4f})")

model = lgb.LGBMRegressor(**PARAMS)
model.fit(train[FEATURES], train[Y], init_score=np.log(train[N]) + c)

# ---------------- 예측 ----------------
raw = model.predict(test[FEATURES], raw_score=True)    # init_score는 예측에 자동 반영되지 않음
pred = np.exp(np.clip(raw + np.log(test[N]) + c, -30, 30))
actual = test[Y].to_numpy(float)


# ---------------- 평가 ----------------
def poisson_dev(y, p):
    p = np.maximum(p, 1e-10)
    t = np.where(y > 0, y * np.log(np.where(y > 0, y, 1) / p), 0.0)
    return 2 * np.mean(t - (y - p))


def metrics(y, p):
    return {"MAE": np.abs(y - p).mean(),
            "RMSE": np.sqrt(((y - p) ** 2).mean()),
            "PoissonDev": poisson_dev(y, p),
            "총량비(예측/실제)": p.sum() / y.sum(),
            "실제합": y.sum(), "예측합": p.sum()}


baseline = test["읍면동업태_분기폐업수"].to_numpy(float)   # 기준선: 이번 분기 폐업수 그대로
res = pd.DataFrame({"B2 모델": metrics(actual, pred),
                    "기준선(전분기 폐업수)": metrics(actual, baseline)})
print("\n=== 2026 Test 성능 (1회 평가) ===")
print(res.round(4))

result = test[["기준분기", "군구명_개편후", "읍면동_개편후", "업태구분명", N, Y]].copy()
result["예측_폐업수"] = pred
result["기준선_폐업수"] = baseline
result["절대오차"] = np.abs(result[Y] - result["예측_폐업수"])
result["점포규모구간"] = pd.cut(result[N], [0, 30, 100, 300, np.inf],
                          labels=["≤30", "31~100", "101~300", ">300"])


def summarize(g):
    return pd.Series({"행수": len(g), "실제평균": g[Y].mean(), "예측평균": g["예측_폐업수"].mean(),
                      "MAE(모델)": g["절대오차"].mean(),
                      "MAE(기준선)": (g[Y] - g["기준선_폐업수"]).abs().mean()})


by_size = result.groupby("점포규모구간", observed=True).apply(summarize)
by_ind = result.groupby("업태구분명").apply(summarize)
by_q = result.groupby("기준분기").apply(summarize)
print("\n=== 점포 규모별 ==="); print(by_size.round(3))
print("\n=== 업태별 ===");     print(by_ind.round(3))
print("\n=== 분기별 ===");     print(by_q.round(3))

# ---------------- SHAP (학습구간 기준) ----------------
contrib = model.predict(train[FEATURES], pred_contrib=True)
shap_vals, base_val = contrib[:, :-1], contrib[:, -1]
recon = shap_vals.sum(1) + base_val
assert np.allclose(recon, model.predict(train[FEATURES], raw_score=True), atol=1e-5), "SHAP 가산성 불일치"

imp = pd.DataFrame({"Feature": FEATURES, "MeanAbsSHAP": np.abs(shap_vals).mean(0)})
imp["Importance_pct"] = imp["MeanAbsSHAP"] / imp["MeanAbsSHAP"].sum() * 100
imp = imp.sort_values("Importance_pct", ascending=False).reset_index(drop=True)
imp["Cumulative_pct"] = imp["Importance_pct"].cumsum()
print("\n=== SHAP 중요도 (B2, 학습구간) ===")
print(imp.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 7))
d = imp.sort_values("Importance_pct")
ax.barh(d["Feature"], d["Importance_pct"])
ax.set_xlabel("Mean |SHAP| (%)"); ax.set_title("B2 Offset Poisson 변수 중요도 (위험도 보정값 기준)")
plt.tight_layout(); plt.savefig(OUT / "B2_SHAP_Bar.png", dpi=200); plt.close()

# ---------------- 저장 ----------------
res.to_csv(OUT / "B2_Test_성능.csv", encoding="utf-8-sig")
by_size.to_csv(OUT / "B2_Test_규모별.csv", encoding="utf-8-sig")
by_ind.to_csv(OUT / "B2_Test_업태별.csv", encoding="utf-8-sig")
by_q.to_csv(OUT / "B2_Test_분기별.csv", encoding="utf-8-sig")
result.to_csv(OUT / "B2_Test_예측결과.csv", index=False, encoding="utf-8-sig")
imp.to_csv(OUT / "B2_SHAP_중요도.csv", index=False, encoding="utf-8-sig")
model.booster_.save_model(str(OUT / "B2_final_model.txt"))
print("\n저장 폴더:", OUT.resolve())